# CNN runs → `runs/` folders (mouse vs Sensorium)

Same visual-only CNN (`PredictorTransfer`, shifter off by default) on either dataset. Every run writes `runs/<timestamp>_<name>/` with `config.json`, `metrics.csv`, `summary.json`, `best_val.pth` and per-neuron arrays. Plot them with `plot_runs.ipynb`.

- **Mouse:** `mouse_time_block_split`: ~12 s blocks go wholly to train or val, with a gap around val blocks.
- **Sensorium:** `SensoriumCachedDataset` (one sample per real 36×64 frame → 60×80) + `sensorium_trial_split`: whole trials go to train or val, over **all** trials (competition tiers ignored).
- Training-size sweeps are specified in **minutes**; subsets take whole blocks / trials, and the val set is the same for every size.
- Small training sets are re-shuffled within an epoch up to `MIN_SAMPLES_PER_EPOCH`, so every run validates every ~100 gradient steps and early stopping is in comparable units; size isn't confounded with the number of updates.
- Headline metric is per-neuron validation correlation (mean over neurons); bits/spike is also logged.

In [ ]:
import os

import numpy as np
import torch

from mouse_model.cnn_predictor_transfer import (
    PredictorTransfer,
    freeze_encoder_backbone,
    load_encoder_backbone_from_checkpoint,
)
from mouse_model.run_logging import RunLogger, load_runs
from mouse_model.sensorium_cache import SensoriumCachedDataset, build_sensorium_cache
from mouse_model.splits import MOUSE_FRAME_S, mouse_time_block_split, sensorium_trial_split
from mouse_model.transfer_cnn_training import train_cnn_transfer

SENSORIUM_SESSIONS = [
    "dynamic29515-10-12-Video-9b4f6a1a067fe51e15306b9628efea20",
    "dynamic29623-4-9-Video-9b4f6a1a067fe51e15306b9628efea20",
    "dynamic29647-19-8-Video-9b4f6a1a067fe51e15306b9628efea20",
    "dynamic29712-5-9-Video-9b4f6a1a067fe51e15306b9628efea20",
    "dynamic29755-2-8-Video-9b4f6a1a067fe51e15306b9628efea20",
]
SENSORIUM_DATA_DIR = os.path.expanduser("~/Downloads")
MOUSE_SESSIONS = ["070921_J553RT", "101521_J559NC", "110421_J569LT"]
DEFAULT_SESSION = {"mouse": MOUSE_SESSIONS[0], "sensorium": SENSORIUM_SESSIONS[0]}

SEED = 0
BATCH_SIZE = 256
LEARNING_RATE = 1e-4
MAX_EPOCHS = 200
PATIENCE = 10
SELECT_BY = "val_cor"  # checkpoint + early stopping on mean val correlation (Poisson val loss diverges first)
MIN_SAMPLES_PER_EPOCH = 100 * BATCH_SIZE  # small training sets are re-shuffled up to this per epoch
VAL_FRAC = 0.3
MOUSE_BLOCK_LEN = 250  # samples (12 s at 48 ms)
SENSORIUM_RESPONSE_LAG = 2  # frames (67 ms); ridge lag sweep peaks at 2-3, mouse uses predict_offset=1 (48 ms)
USE_SHIFTER = False
NUM_WORKERS = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

_sensorium = {}


def sensorium_session(session):
    """Cached dataset for one Sensorium session; the cache is built once (~2 min, ~5 GB)."""
    if session not in _sensorium:
        cache_dir = build_sensorium_cache(os.path.join(SENSORIUM_DATA_DIR, session))
        _sensorium[session] = (SensoriumCachedDataset(cache_dir, response_lag=SENSORIUM_RESPONSE_LAG), cache_dir)
    return _sensorium[session]


ds, _ = sensorium_session(DEFAULT_SESSION["sensorium"])
print("Sensorium frames:", len(ds), f"({len(ds) / ds.fps / 60:.0f} min)")

In [ ]:
def minutes_to_samples(dataset, minutes):
    if minutes is None:
        return None
    per_min = 60.0 / MOUSE_FRAME_S if dataset == "mouse" else 60.0 * ds.fps
    return int(round(minutes * per_min))


def load_data(dataset, session, max_train_samples, seed):
    if dataset == "mouse":
        tr, va, info = mouse_time_block_split(
            session, block_len=MOUSE_BLOCK_LEN, val_frac=VAL_FRAC, seed=seed,
            max_train_samples=max_train_samples,
        )
        return tr, va, info, info["num_neurons"]
    sds, cache_dir = sensorium_session(session)
    tr, va, info = sensorium_trial_split(
        sds, val_frac=VAL_FRAC, seed=seed, max_train_samples=max_train_samples,
    )
    info["response_lag"] = SENSORIUM_RESPONSE_LAG
    info["cache_dir"] = cache_dir
    return tr, va, info, sds.num_neurons


def already_done(**cfg):
    return any(r["summary"] for r in load_runs(**cfg))


def run_one(dataset, train_minutes=None, seed=SEED, training_mode="from_scratch",
            pretrained_run_dir=None, skip_existing=True, session=None):
    session = session or DEFAULT_SESSION[dataset]
    max_train = minutes_to_samples(dataset, train_minutes)
    key = dict(dataset=dataset, session=session, train_minutes_requested=train_minutes, seed=seed,
               training_mode=training_mode, shifter=USE_SHIFTER, select_by=SELECT_BY,
               pretrained_run=None if pretrained_run_dir is None else os.path.basename(pretrained_run_dir))
    if skip_existing and already_done(**key):
        print("skip (done):", key)
        return None

    tr, va, info, n_neurons = load_data(dataset, session, max_train, seed)
    model = PredictorTransfer(num_neurons=n_neurons, use_shifter=USE_SHIFTER).to(device)
    frozen = training_mode == "transfer_frozen_backbone"
    if pretrained_run_dir is not None:
        sd = torch.load(os.path.join(pretrained_run_dir, "best_val.pth"), map_location=device)
        load_encoder_backbone_from_checkpoint(model, sd)
        if frozen:
            freeze_encoder_backbone(model)

    tag = "full" if train_minutes is None else f"{train_minutes}min"
    short = session.split("-")[0] if dataset == "sensorium" else session.split("_")[-1]
    run = RunLogger(
        f"{dataset}_{short}_{training_mode}_{tag}_s{seed}",
        {**key, "model": "cnn", "max_train_samples": max_train, "split_info": info},
    )
    print(run.dir, f"| train {info['n_train']} ({info['train_minutes']:.1f} min) val {info['n_val']}")
    train_cnn_transfer(
        model, device, tr, va,
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
        seed=seed, num_workers=NUM_WORKERS, freeze_encoder_backbone=frozen, patience=PATIENCE,
        min_samples_per_epoch=MIN_SAMPLES_PER_EPOCH, select_by=SELECT_BY, run=run,
    )
    return run.dir

## Learning-curve sweep (scratch)

Mouse `070921_J553RT` has ~20 min of training data under this split; one Sensorium session has ~80 min. `None` = all training data. Re-running the cell skips finished runs.

In [ ]:
TRAIN_MINUTES = {
    "mouse": [1, 2, 5, 10, None],
    "sensorium": [1, 2, 5, 10, 20, 40, None],
}
SEEDS = [0]

for seed in SEEDS:
    for dataset, minutes_list in TRAIN_MINUTES.items():
        for minutes in minutes_list:
            run_one(dataset, minutes, seed=seed)

## Transfer (backbone from a finished full-data scratch run)

In [ ]:
def full_scratch_run(dataset, seed=SEED):
    runs = [r for r in load_runs(dataset=dataset, session=DEFAULT_SESSION[dataset],
                                 training_mode="from_scratch", seed=seed,
                                 train_minutes_requested=None, select_by=SELECT_BY) if r["summary"]]
    return runs[-1]["dir"] if runs else None


for target, source in [("sensorium", "mouse"), ("mouse", "sensorium")]:
    src = full_scratch_run(source)
    if src is None:
        print(f"no finished full {source} scratch run yet")
        continue
    for mode in ["transfer_frozen_backbone", "transfer_finetune"]:
        run_one(target, None, training_mode=mode, pretrained_run_dir=src)

## Replication: other animals and seeds

Same scratch learning curve on the other two mice and the other four Sensorium sessions (seed 0), plus seeds 1-2 on the default mouse. Minutes are the outer loop for Sensorium so a partial sweep still gives every session the small sizes.


In [ ]:
for session in MOUSE_SESSIONS[1:]:
    for minutes in TRAIN_MINUTES["mouse"]:
        run_one("mouse", minutes, seed=0, session=session)

for seed in [1, 2]:
    for minutes in TRAIN_MINUTES["mouse"]:
        run_one("mouse", minutes, seed=seed)

for minutes in TRAIN_MINUTES["sensorium"]:
    for session in SENSORIUM_SESSIONS[1:]:
        run_one("sensorium", minutes, seed=0, session=session)
